# 02 - Análise Exploratória de Dados (EDA)

Este notebook realiza a Análise Exploratória de Dados (EDA) da base de notas fiscais do projeto `model_supermarket`, utilizando os módulos analíticos desenvolvidos na arquitetura do framework.

O objetivo é explorar padrões de consumo, comportamento temporal, distribuição de gastos, frequência de produtos e indicadores gerais das compras realizadas em supermercados.

As análises incluem:

- validação estrutural da base
- resumos estatísticos
- métricas gerais de consumo
- análises temporais
- agregações por supermercado
- análises por produto
- histórico de preços
- identificação de padrões de compra
- preparação para visualização em dashboard e modelos estatísticos

In [1]:
# !pip install -r requirements.txt

In [2]:
# IMPORTS
from pathlib import Path
import time
import openpyxl
import pandas as pd
from pandas import read_excel
from src.core.config_loader import ConfigLoader
from src.core.tictoc import tictoc
from src.analysis.eda import *

# CONFIGURAÇÃO
tic_geral = time.time()
config = ConfigLoader("config/config.yaml")
raw_path = Path(config.get('data.raw_invoice_path'))

## 1. Carregamento dos Dados

| Variável               | Tipo da Variável           | Descrição                                                                   |
| ---------------------- | -------------------------- | --------------------------------------------------------------------------- |
| `CHAVE_ANONIMIZADA`    | Chave                      | Identificador único anonimizado da NFC-e utilizando hash SHA256.            |
| `DATA`                 | Data                       | Data e horário de emissão da nota fiscal no formato `DD/MM/AAAA HH:MM:SS`.  |
| `MES_ANO`              | Categórico                 | Representação da competência da compra no formato `AAAAMM`.                 |
| `PERIODO`              | Categórico                 | Período do dia em que a compra foi realizada (`MANHA`, `TARDE` ou `NOITE`). |
| `CNPJ`                 | Chave                      | CNPJ do estabelecimento emissor da nota fiscal.                             |
| `SUPERMERCADO`         | Categórico                 | Nome do supermercado onde a compra foi realizada.                           |
| `QTDE_TOTAL_NOTA`      | Numérico Inteiro           | Quantidade total de itens presentes na nota fiscal.                         |
| `VALOR_TOTAL_NOTA`     | Numérico Contínuo          | Valor total da nota fiscal.                                                 |
| `VALOR_TOTAL_TRIBUTOS` | Numérico Contínuo          | Valor total estimado de tributos incidentes sobre a compra.                 |
| `COD_PRODUTO`          | Chave                      | Código identificador único do produto.                                      |
| `CAT_PRODUTO`          | Categórico                 | Categoria associada ao produto.                                             |
| `PRODUTO`              | Categórico                 | Nome/descrição padronizada do produto.                                      |
| `UNIDADE`              | Categórico                 | Unidade de medida ou comercialização do produto.                            |
| `QTDE`                 | Numérico Contínuo          | Quantidade adquirida do produto na nota fiscal.                             |
| `VALOR_PRODUTO`        | Numérico Contínuo          | Valor unitário do produto.                                                  |
| `VALOR_TOTAL_PRODUTO`  | Numérico Contínuo          | Valor total pago pelo produto (`QTDE × VALOR_PRODUTO`).                     |


In [3]:
tic = time.time()

In [4]:
df = read_excel('data/notas_fiscais_supermercado.xlsx')
df.head()

,chave_anonimizada,data_hora,mes_ano,dia_semana,feriado,estacao_ano,temperatura_max,temperatura_min,temperatura_media,chuva_mm,...,qtd_total_nota,valor_total_nota,valor_total_tributos,cod_produto,categoria_produto,produto,unidade,quantidade,preco_unitario,preco_total
0,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,10176020001,CEREAL,CEREAL NESCAU 770G,UNIDADE,1.0,19.90,19.90
1,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,11421800001,MERCEARIA,BEB VERY 1250G SALAD,UNIDADE,1.0,7.70,7.70
2,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,18730001,MERCEARIA,ARR T JOAO LF T1 5kg,UNIDADE,1.0,25.90,25.90
3,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,65420001,BEBIDA NAO ALCOOLICA,BEB NESQUIK 200ML MO,UNIDADE,10.0,2.89,28.90
4,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,10566170001,MOLHO,EXT ELEF TP 140G,UNIDADE,2.0,2.59,5.18


In [5]:
toc = time.time()
print(f"\n\033[33mEtapa 1 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 1 | Tempo: (00:00:01) [19/05/2026 02:21:23 - [19/05/2026 02:21:25]


## 2. Análise

**🔹 Resumos Gerais**

* Quantidade total de produtos distintos
* Quantidade total de códigos distintos
* Verificação de inconsistências entre códigos e nomes
* Total gasto em compras
* Quantidade de notas fiscais
* Média de gasto por compra

**🔹 Análises por Categoria Temporal**

* Resumo de gastos por:

  * supermercado
  * período do dia
  * ano
  * mês
* Evolução temporal dos gastos
* Frequência de compras ao longo do tempo

**🔹 Análises por Produto**

* Identificação do produto mais comprado
* Consulta detalhada por código do produto
* Consulta textual por nome do produto
* Histórico de preços dos produtos
* Frequência de compra por item

### 2.1 Resumos Gerais

In [6]:
prod_unicos = produtos_unicos(df)

Antes da transformação:
Há 1087 código(s) de produto(s) distinto(s).
Há 1087 nome(s) de produto(s) distinto(s).


Após a transformação:
Há 1087 código(s) de produto(s) distinto(s).
Há 1087 nome(s) de produto(s) distinto(s).
Há 0 código(s) de produto(s) com nomes diferentes


In [7]:
resumo_val_gerais = resumo_valores_gerais(df)

Resumos gerais:
Total de gastos: R$ 1334099.4
Quantidade de idas ao mercado: 177
Média de gastos por nota fiscal: 7537.28


### 2.2 Análises por Categoria Temporal

In [8]:
resumo_supermercado, resumo_periodo = resumo_valores_lugar_periodo(df)


Resumo por supermercado:
supermercado
ASSAI       176335.49
CONDOR       16732.61
HONESTY       1268.73
MAX        1139762.57
Name: valor_total_nota, dtype: float64

Resumo por período:
periodo_dia
MADRUGADA         34.41
MANHA           8883.94
NOITE        1136682.25
TARDE         188498.80
Name: valor_total_nota, dtype: float64


In [9]:
resumo_total_por_ano, resumo_total_por_mes = resumo_valores_ano_mes(df)


Resumo por ano:
data_hora
2022    187095.53
2023    304254.21
2024    263758.37
2025    475134.13
2026    103857.16
Name: valor_total_nota, dtype: float64

Resumo por mês:
data_hora
2022-06     55742.78
2022-07      9446.46
2022-08       201.30
2022-09     92614.64
2022-10      7061.10
2022-11      8665.47
2022-12     13363.78
2023-01     10356.81
2023-02     48593.00
2023-03      7303.01
2023-04      8797.82
2023-05      5171.73
2023-06     23919.06
2023-07     52023.50
2023-08     41874.73
2023-09     67603.72
2023-10     27730.75
2023-11      1157.55
2023-12      9722.53
2024-01     40448.41
2024-02     31574.77
2024-03     16619.54
2024-04     17705.49
2024-05      1410.29
2024-06      3390.92
2024-07      3825.02
2024-08    112407.31
2024-10     11427.48
2024-11     18428.15
2024-12      6520.99
2025-01     96619.69
2025-02     84356.57
2025-03       455.21
2025-04     80201.94
2025-05      7823.94
2025-06     39001.77
2025-08     11106.34
2025-09     21873.47
2025-10     96422.9

### 2.2 Análises por Produto

In [10]:
produto_mais_comprado = resumo_produto_mais_comprado(df)


Resumo do produto mais comprado:
O produto que mais comprei foi: AGUA CRIST AZ 510ML


In [11]:
exibe_subtabela_cod_produto(df, 940360001)

,cod_produto,produto,preco_unitario,data_hora
50,940360001,TEKITOS 300G TRAD,8.99,2022-06-28 20:44:50
301,940360001,TEKITOS 300G TRAD,8.99,2022-11-09 11:36:52
327,940360001,TEKITOS 300G TRAD,8.99,2022-11-18 21:41:27
366,940360001,TEKITOS 300G TRAD,8.99,2022-12-06 21:20:11
430,940360001,TEKITOS 300G TRAD,8.99,2022-12-18 10:19:36
454,940360001,TEKITOS 300G TRAD,8.99,2023-01-15 18:17:47
621,940360001,TEKITOS 300G TRAD,8.99,2023-03-14 20:38:05
1140,940360001,TEKITOS 300G TRAD,10.49,2023-09-14 18:26:01


In [12]:
exibe_subtabela_nome_produto(df, 'whisky')

,produto,cod_produto,preco_unitario,data_hora
592,WHISKY PASSP 670ML,279691,48.99,2023-03-01 17:50:26
649,WHISKY PASSP 670ML,279691,47.99,2023-04-08 16:59:59
688,WHISKY PASSP 670ML,279691,47.99,2023-04-20 18:34:11
1646,WHISKY PASSP 670ML,279691,61.99,2024-03-06 20:31:25
1790,WHISKY PASSP 670ML,279691,58.99,2024-04-29 16:44:08
1986,WHISKY PASSP 670ML,279691,49.99,2024-08-25 14:20:05
2007,WHISKY PASSP 670ML,279691,57.99,2024-10-18 21:33:44
2094,WHISKY PASSP 670ML,279691,57.99,2024-11-28 17:17:03
2159,WHISKY PASSP 670ML,279691,49.99,2024-12-24 15:25:31
2187,WHISKY PASSP 670ML,279691,57.99,2025-01-06 21:42:46


In [13]:
toc = time.time()
print(f"\n\033[33mEtapa 2 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 2 | Tempo: (00:00:03) [19/05/2026 02:21:23 - [19/05/2026 02:21:27]


## 3. Tempo Decorrido

In [14]:
toc_geral = time.time()
print(f"\n\033[33mTempo decorrido no notebook:\033[0;0m {tictoc(tic_geral, toc_geral)}")


Tempo decorrido no notebook: (00:00:03) [19/05/2026 02:21:23 - [19/05/2026 02:21:27]
